# EVA02-L/14: domain-balanced sampling for UPAR Track 1

Run all cells in Kaggle with **GPU and Internet enabled**. This notebook adapts the strongest recorded single model, EVA02-L/14 (hidden-test Avg 0.7825), not the two-model ensemble (0.7856). No result is claimed for this new experiment.

The default samples each training image with probability `0.5 / N + 0.5 / (K * n_domain)`: half image-uniform and half domain-uniform. Counts are read from the training annotations, never inferred from hidden-test inputs. The backbone, global head, five epochs, batch 16, 224-square input, optimizer, and augmentation follow the EVA02-L baseline. An epoch has `floor(N / batch)` optimizer steps in every mode.

For the primary comparison, run `SAMPLING_MODE = "image"`, preserve its outputs, then restart and run `"mixture"` from the same generic pretrained initialization. Both use flip-averaged validation and fixed image-distribution loss weights. Do not compare against the historical score as if it were a matched control: the old notebook selected checkpoints with single-view validation. Mixture sampling also uses replacement; `"image_replacement"` provides an additional control for that change. `"domain"` is an optional fully balanced arm.

The download cells retain the original dataset handling. Training requires substantial GPU time and disk space. Only the organizer-provided training split is used for fine-tuning. Validation selects checkpoints; no hidden-test data is read. Outputs include configuration, annotation hashes, per-domain metrics and sample counts, validation logits, checkpoint, and an offline submission ZIP.

**Quota recovery:** use Kaggle **Add Input** to attach source images, original ZIPs, or a previous successful run's saved data. Inputs under `/kaggle/input` are reused automatically before contacting download hosts. A Drive quota block cannot be bypassed by changing notebook packages; without a local copy, wait for the host quota to reset.


## 1. Download challenge data

These cells are copied from `UPAR-Challenge-2027/kaggle_download_datasets.ipynb`. They download Market1501, PA-100K, and PETA and copy the challenge annotations into `/kaggle/working/upar_2027/data`. Source dataset licenses and usage restrictions apply.


In [ ]:
%pip -q install gdown tqdm requests timm==1.0.22

In [ ]:
import csv
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import gdown
import requests
from tqdm.auto import tqdm

WORK = Path('/kaggle/working')
REPO = WORK / 'UPAR-Challenge-2027'
DATA = WORK / 'upar_2027' / 'data'
DATA.mkdir(parents=True, exist_ok=True)

# Attach a dataset or a previous successful notebook's saved data using
# Kaggle's Add Input. Both extracted images and original ZIPs are supported.
# Add custom cache directories here if they are outside /kaggle/input.
LOCAL_DATA_ROOTS = [Path('/kaggle/input')]
if os.environ.get('UPAR_DATA_ROOT'):
    LOCAL_DATA_ROOTS.append(Path(os.environ['UPAR_DATA_ROOT']))

if not (REPO / 'peta_file_mapping.txt').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/speckean/UPAR-Challenge-2027.git', str(REPO)
    ], check=True)
for split in ('train', 'val'):
    destination = DATA / 'annotations' / f'{split}.csv'
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO / 'data' / 'annotations' / f'{split}.csv', destination)
print('Data directory:', DATA)

In [ ]:
"""Local UPAR image/ZIP reuse helpers; embedded in the standalone notebook."""
import os
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path, PurePosixPath


def portable_path(value):
    return str(value).replace("\\", "/").strip("/")


def index_local_assets(roots):
    """Scan attached inputs once, without following directory symlinks."""
    images = defaultdict(list)
    archives = []
    seen = set()
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for directory, _, filenames in os.walk(root, followlinks=False):
            for filename in filenames:
                path = Path(directory) / filename
                key = str(path.absolute())
                if key in seen:
                    continue
                seen.add(key)
                suffix = path.suffix.lower()
                if suffix in {".jpg", ".jpeg", ".png", ".bmp"}:
                    images[filename].append(path)
                elif suffix == ".zip":
                    archives.append(path)
    return images, sorted(archives)


def source_variants(relative, peta_mapping):
    relative = portable_path(relative)
    source, remainder = relative.split("/", 1)
    variants = [relative]
    if source == "Market1501":
        variants.extend([f"Market-1501-v15.09.15/{remainder}", remainder])
    elif source == "PA100k":
        variants.extend([f"PA100K/{remainder}", f"PA-100K/{remainder}"])
        if remainder.startswith("release_data/"):
            variants.append(remainder)
        else:
            variants.append(f"release_data/{remainder}")
    elif source == "PETA" and relative in peta_mapping:
        original = portable_path(peta_mapping[relative])
        variants.append(original)
        variants.append(original.removeprefix("PETA/"))
    return list(dict.fromkeys(variants))


def choose_suffix_match(candidates, variants):
    # Prefer the full challenge path, then known original archive layouts.
    # Never guess by basename alone: PETA originals can share basenames.
    for variant in variants:
        matches = [candidate for candidate in candidates
                   if portable_path(candidate) == variant
                   or portable_path(candidate).endswith("/" + variant)]
        if matches:
            return sorted(matches, key=str)[0]
    return None


def reuse_local_source(data_root, relative_paths, image_index, archives, peta_mapping):
    """Copy only missing annotated images; never import external labels."""
    data_root = Path(data_root)
    relative_paths = list(relative_paths)
    for relative in relative_paths:
        value = str(relative).replace("\\", "/")
        if PurePosixPath(value).is_absolute() or ".." in PurePosixPath(value).parts or ":" in value:
            raise ValueError(f"Unsafe annotation path: {relative!r}")
    absent = {portable_path(path) for path in relative_paths
              if not (data_root / portable_path(path)).is_file()}
    copied = 0
    for relative in sorted(absent):
        variants = source_variants(relative, peta_mapping)
        candidates = []
        for basename in {PurePosixPath(variant).name for variant in variants}:
            candidates.extend(image_index.get(basename, []))
        original = choose_suffix_match(candidates, variants)
        if original is not None:
            target = data_root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, target)
            absent.remove(relative)
            copied += 1
    for archive_path in archives:
        if not absent:
            break
        try:
            with zipfile.ZipFile(archive_path) as archive:
                members = defaultdict(list)
                for info in archive.infolist():
                    if not info.is_dir():
                        members[PurePosixPath(portable_path(info.filename)).name].append(info.filename)
                for relative in sorted(absent):
                    variants = source_variants(relative, peta_mapping)
                    candidates = []
                    for basename in {PurePosixPath(variant).name for variant in variants}:
                        candidates.extend(members.get(basename, []))
                    member = choose_suffix_match(candidates, variants)
                    if member is None:
                        continue
                    # Destination comes only from validated annotations, never ZIP paths.
                    target = data_root / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_name(target.name + ".part")
                    with archive.open(member) as original, temporary.open("wb") as output:
                        shutil.copyfileobj(original, output)
                    temporary.replace(target)
                    absent.remove(relative)
                    copied += 1
        except (zipfile.BadZipFile, EOFError) as error:
            print(f"Skipping invalid local ZIP {archive_path}: {error}")
    return copied


def local_reuse_self_check():
    """Small real filesystem checks, before any network access."""
    import tempfile
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        inputs, output = root / "inputs", root / "output"
        image = inputs / "old-output/upar_2027/data/Market1501/bounding_box_train/a.jpg"
        image.parent.mkdir(parents=True)
        image.write_bytes(b"market-fixture")
        with zipfile.ZipFile(inputs / "originals.zip", "w") as archive:
            archive.writestr("release_data/b.jpg", b"pa-fixture")
            archive.writestr("PETA/group/0001.png", b"peta-fixture")
            archive.writestr("PETA/other/0001.png", b"wrong-peta")
            archive.writestr("../../outside.txt", b"unsafe-unused-member")
        images, archives = index_local_assets([inputs])
        paths = ["Market1501/bounding_box_train/a.jpg", "PA100k/release_data/b.jpg", "PETA/mapped.png"]
        mapping = {"PETA/mapped.png": "PETA/group/0001.png"}
        assert reuse_local_source(output, paths, images, archives, mapping) == 3
        assert [(output / path).read_bytes() for path in paths] == [b"market-fixture", b"pa-fixture", b"peta-fixture"]
        assert reuse_local_source(output, paths, images, archives, mapping) == 0
        assert not (root / "outside.txt").exists()
        assert choose_suffix_match(["unrelated/a.jpg"], ["bounding_box_train/a.jpg"]) is None
        with zipfile.ZipFile(inputs / "market.zip", "w") as archive:
            archive.writestr("Market-1501-v15.09.15/query/c.jpg", b"original-market")
        images, archives = index_local_assets([inputs])
        assert reuse_local_source(output, ["Market1501/query/c.jpg"], images, archives, {}) == 1
        assert (output / "Market1501/query/c.jpg").read_bytes() == b"original-market"
    print("Local image/ZIP reuse, PETA mapping, rerun and path checks passed")


# Only annotation-referenced images are reused. Keep organizer labels unchanged.
needed = {'Market1501': set(), 'PA100k': set(), 'PETA': set()}
for split in ('train', 'val'):
    with (DATA / 'annotations' / f'{split}.csv').open(newline='') as handle:
        reader = csv.reader(handle)
        next(reader)
        for row in reader:
            value = row[0].replace(chr(92), '/')
            relative = PurePosixPath(value)
            if relative.is_absolute() or '..' in relative.parts or ':' in value:
                raise ValueError(f'Unsafe annotation image path: {value!r}')
            if relative.parts[0] not in needed:
                raise ValueError(f'Unexpected source in annotation: {value!r}')
            needed[relative.parts[0]].add(Path(value))

def missing(source):
    return [path for path in needed[source] if not (DATA / path).is_file()]

def check_zip(path):
    # Preserve invalid cached files for diagnosis; downloads use .part files.
    return path.is_file() and zipfile.is_zipfile(path)

def require_complete(source):
    absent = missing(source)
    if absent:
        raise RuntimeError(f'{source}: {len(absent)} images missing; examples: {absent[:5]}. '
                           'Attach complete source images or an original ZIP using Kaggle Add Input.')
    print(f'{source}: {len(needed[source]):,} annotated images available')

with (REPO / 'peta_file_mapping.txt').open(newline='') as handle:
    peta_mapping = {portable_path(destination): portable_path(original)
                    for original, destination in csv.reader(handle)}

local_reuse_self_check()
local_images, local_archives = index_local_assets(LOCAL_DATA_ROOTS)
print('Attached local images:', sum(map(len, local_images.values())), 'ZIPs:', len(local_archives))
print('Required image counts:', {source: len(paths) for source, paths in needed.items()})

def restore_attached(source):
    if missing(source):
        copied = reuse_local_source(DATA, needed[source], local_images, local_archives, peta_mapping)
        print(f'{source}: reused {copied:,} local images; {len(missing(source)):,} still missing')

def download_failure(source, error):
    return RuntimeError(
        f'{source} download is unavailable: {error}\n\n'
        'For the Google Drive "too many users" message, the file owner\'s download quota '
        'is exhausted; changing debugger settings or reinstalling gdown will not resolve it. '
        'In Kaggle, use Add Input to attach a dataset or a previous successful notebook\'s '
        'saved output containing the source images or original ZIP. Keep the original '
        'dataset folders or ZIP layout. This notebook scans /kaggle/input automatically '
        'and copies only annotation-referenced images into /kaggle/working/upar_2027/data. '
        'Then rerun from the first cell. If no cache is available, retry when the host '
        'quota resets. Training cannot proceed with missing images.'
    )

def download_zip(url, destination, source):
    temporary = destination.with_name(destination.name + '.part')
    try:
        result = gdown.download(url, output=str(temporary), quiet=False,
                                use_cookies=False, fuzzy=True)
        if not result or not zipfile.is_zipfile(temporary):
            raise RuntimeError('Host did not return a valid ZIP archive')
        temporary.replace(destination)
    except Exception as error:
        raise download_failure(source, error) from error

## Market1501: reuse attached data before downloading

The public Google Drive file may be quota-limited. Attach a previous successful notebook's saved dataset or the original Market1501 ZIP via **Add Input**. The notebook recognizes both `Market1501/...` and `Market-1501-v15.09.15/...` layouts, including renamed ZIPs. If no local copy is available, it tries the original download once and explains recovery if the host refuses it. Debugger warnings are unrelated.

In [ ]:
market_archive = DATA / 'market_1501.zip'
restore_attached('Market1501')
if missing('Market1501'):
    if not check_zip(market_archive):
        download_zip(
            'https://drive.google.com/file/d/0B8-rUzbwVRk0c054eEozWG9COHM/view?resourcekey=0-8nyl7K9_x37HlQm34MmrYQ',
            market_archive, 'Market1501',
        )
    reused = reuse_local_source(DATA, needed['Market1501'], {}, [market_archive], peta_mapping)
    print(f'Market1501: restored {reused:,} images from downloaded/cached ZIP')
require_complete('Market1501')
# Keep cached archives so reruns need not hit the quota-limited host again.

## PA-100K

In [ ]:
pa_dir = DATA / 'PA100k'
pa_dir.mkdir(exist_ok=True)
pa_archive = pa_dir / 'data.zip'
restore_attached('PA100k')
if missing('PA100k'):
    if not check_zip(pa_archive):
        try:
            gdown.download_folder(
                'https://drive.google.com/drive/folders/1d_D0Yh7C262gr0ef9EqkvG_M3fqgAWa2?usp=sharing',
                output=str(pa_dir), quiet=False, use_cookies=False,
            )
            if not check_zip(pa_archive):
                raise RuntimeError('PA-100K data.zip was not downloaded as a valid ZIP')
        except Exception as error:
            raise download_failure('PA100k', error) from error
    reused = reuse_local_source(DATA, needed['PA100k'], {}, [pa_archive], peta_mapping)
    print(f'PA100k: restored {reused:,} images from downloaded/cached ZIP')
require_complete('PA100k')

## PETA

Only the mapped images are extracted from the archive, directly to their challenge filenames.

In [ ]:
peta_dir = DATA / 'PETA'
peta_dir.mkdir(exist_ok=True)
peta_archive = peta_dir / 'peta.zip'
restore_attached('PETA')
if missing('PETA'):
    if not check_zip(peta_archive):
        temporary = peta_archive.with_name(peta_archive.name + '.part')
        try:
            url = 'https://www.dropbox.com/s/52ylx522hwbdxz6/PETA.zip?dl=1'
            with requests.get(url, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                total = int(response.headers.get('content-length', 0))
                with temporary.open('wb') as output, tqdm(total=total or None, unit='B', unit_scale=True, desc='PETA') as bar:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            output.write(chunk)
                            bar.update(len(chunk))
            if not zipfile.is_zipfile(temporary):
                raise RuntimeError('PETA host did not return a valid ZIP')
            temporary.replace(peta_archive)
        except Exception as error:
            raise download_failure('PETA', error) from error
    reused = reuse_local_source(DATA, needed['PETA'], {}, [peta_archive], peta_mapping)
    print(f'PETA: restored {reused:,} mapped images from downloaded/cached ZIP')
require_complete('PETA')

In [ ]:
for source in needed:
    require_complete(source)
print('Ready:', DATA)
print('Train annotations:', DATA / 'annotations' / 'train.csv')
print('Validation annotations:', DATA / 'annotations' / 'val.csv')

## 2. Matched training configuration

Default: `SAMPLING_MODE = "mixture"`, `DOMAIN_MIX = 0.5`, `LOSS_WEIGHT_MODE = "fixed_image"`. Use `"image"` for the ordinary shuffled control; `"image_replacement"` to control for replacement alone; `"domain"` for equal expected source probabilities. Mixture/domain modes balance in expectation, not within every batch. Repeated images and omitted images are expected with replacement.

Keep five epochs, batch 16, peak LR 5e-5, layer decay 0.85, and seed 42 matched. Both comparison arms validate with flip-logit averaging. The original pretrained EVA02-L checkpoint is interpolated to 224 x 224, as in the existing baseline. Restart the kernel between arms. Mode, mixture fraction, loss-weight mode, and run suffix appear in artifact names. Existing outputs cause training to stop instead of silently overwriting them.

`fixed_image` retains positive weights from the original training image distribution to isolate sampling. `sampling` recomputes them from the sampler's expected distribution and is a separate sampling-plus-loss experiment. Domain labels are used only in training and validation diagnostics; inference has no domain-specific routing.

In [ ]:
"""Kaggle training script for UPAR 2027 Track 1 with a timm EVA02-L/14 (EVA02).

Reads challenge data from /kaggle/working/upar_2027/data by default.
This file is also the source for the training cells in the accompanying notebook.
"""

import csv
import gc
import hashlib
import json
from collections import Counter
import importlib.util
import math
import os
import random
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms


DATA_ROOT = DATA  # Train on the complete writable dataset prepared above.
OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
SAMPLING_MODE = "mixture"  # "image", "mixture", "image_replacement", or "domain".
DOMAIN_MIX = 0.5  # Mixture's domain-uniform fraction; 0=image uniform, 1=equal domains.
LOSS_WEIGHT_MODE = "fixed_image"  # Keep fixed for sampling ablation; optional "sampling".
VALIDATION_FLIP = True  # Identical for every arm; submission always uses flip averaging.
INFERENCE_BATCH_SIZE = 16
RUN_SUFFIX = "v1"  # Change before rerunning an identical configuration.
MODEL_TAG = f"upar_track1_eva02_l14_domain_sampling_{SAMPLING_MODE}_mix{DOMAIN_MIX:g}_{LOSS_WEIGHT_MODE}_{RUN_SUFFIX}"
ARCHITECTURE = "eva02_large_patch14_448"  # timm EVA02
PRETRAINED_TAG = "mim_in22k_ft_in22k_in1k"  # ImageNet-22k MIM pretraining, ImageNet-22k then ImageNet-1k fine-tuning.
EPOCHS = 5
WARMUP_EPOCHS = 1
BATCH_SIZE = 16  # 32 may not fit in 16 GB with 257 tokens per image.
NUM_WORKERS = 4
LEARNING_RATE = 5e-5  # Peak learning rate of the classifier head.
LAYER_DECAY = 0.85  # Each earlier transformer block gets this factor less learning rate.
WEIGHT_DECAY = 0.05
DROP_PATH = 0.2  # Stochastic depth rate.
USE_IMAGENET_WEIGHTS = True  # Needs Internet once (Hugging Face Hub), or a cached timm checkpoint.
IMAGE_SIZE = 224  # 16 x 16 patches of 14 pixels, the grid of EVA02's rotary position embeddings.
MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
STD = (0.26862954, 0.26130258, 0.27577711)
SEED = 42

## 3. Read annotations and images

The released training CSV header omits the image-column label; this parser handles both train and validation formats.

In [ ]:
def read_annotations(path):
    with path.open(newline="") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        # The released train.csv omits the image-column name in its header.
        names = header if len(header) == 40 else header[1:]
        names = [name.removeprefix("# ").strip() for name in names]
        if len(names) != 40:
            raise ValueError(f"Expected 40 attributes in {path}, got {len(names)}")
        paths, labels = [], []
        for row_number, row in enumerate(reader, 2):
            if len(row) != 41:
                raise ValueError(f"{path}:{row_number}: expected image path and 40 labels")
            paths.append(row[0])
            labels.append([int(value) for value in row[1:]])
    labels = np.asarray(labels, dtype=np.float32)
    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"{path} contains labels other than 0 and 1")
    return names, paths, labels


class AttributeDataset(Dataset):
    def __init__(self, root, paths, labels, transform):
        self.root, self.paths, self.labels, self.transform = root, paths, labels, transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.root / self.paths[index]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.from_numpy(self.labels[index])




def source_from_path(path):
    normalized = str(path).replace("\", "/")
    parts = normalized.split("/")
    if len(parts) < 2 or parts[0] not in {"Market1501", "PA100k", "PA100K", "PA-100K", "PETA"}:
        raise ValueError(f"Unexpected source path: {path!r}; inspect annotation prefixes")
    return {"PA100k": "PA100K", "PA-100K": "PA100K"}.get(parts[0], parts[0])


def sampling_distribution(sources, mode, domain_mix):
    if mode not in {"image", "image_replacement", "mixture", "domain"}:
        raise ValueError(f"Unknown sampling mode: {mode}")
    if not 0 <= domain_mix <= 1:
        raise ValueError("DOMAIN_MIX must be in [0, 1]")
    if not sources:
        raise ValueError("No training rows")
    counts = Counter(sources)
    alpha = domain_mix if mode == "mixture" else float(mode == "domain")
    probabilities = np.asarray([
        (1 - alpha) / len(sources) + alpha / (len(counts) * counts[source])
        for source in sources
    ], dtype=np.float64)
    if not np.isfinite(probabilities).all() or not np.isclose(probabilities.sum(), 1):
        raise AssertionError("Invalid sampling distribution")
    return probabilities


class DomainSampler(Sampler):
    """Deterministic epoch draws with an identical update budget in every arm."""
    def __init__(self, sources, mode, domain_mix, batch_size, seed):
        if batch_size <= 0:
            raise ValueError("batch_size must be positive")
        self.sources = list(sources)
        self.mode, self.seed, self.epoch = mode, seed, 0
        self.probabilities = sampling_distribution(self.sources, mode, domain_mix)
        self.num_samples = len(sources) // batch_size * batch_size
        if self.num_samples == 0:
            raise ValueError("Not enough training images for one full batch")
        self.last_counts = {}

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return self.num_samples

    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed + self.epoch)
        if self.mode == "image":
            indices = torch.randperm(len(self.sources), generator=generator)[:self.num_samples]
        else:
            indices = torch.multinomial(
                torch.as_tensor(self.probabilities, dtype=torch.float64),
                self.num_samples, replacement=True, generator=generator,
            )
        indices = indices.tolist()
        self.last_counts = dict(Counter(self.sources[index] for index in indices))
        return iter(indices)


def positive_weights(labels, probabilities, mode):
    if mode == "fixed_image":
        positives = labels.sum(axis=0)
        ratio = (len(labels) - positives) / np.maximum(positives, 1)
    elif mode == "sampling":
        prevalence = probabilities @ labels.astype(np.float64)
        ratio = (1 - prevalence) / np.maximum(prevalence, 1 / len(labels))
    else:
        raise ValueError(f"Unknown LOSS_WEIGHT_MODE: {mode}")
    return np.clip(ratio, 1, 20).astype(np.float32)


def sampler_self_check():
    # Tiny unequal domains: verify probabilities, budget, reproducibility,
    # non-replacement control, and observed replacement proportions.
    sources = ["A"] * 80 + ["B"] * 15 + ["C"] * 5
    expected_masses = {
        "image": [0.8, 0.15, 0.05],
        "image_replacement": [0.8, 0.15, 0.05],
        "mixture": [0.8 / 2 + 1 / 6, 0.15 / 2 + 1 / 6, 0.05 / 2 + 1 / 6],
        "domain": [1 / 3] * 3,
    }
    for mode, expected in expected_masses.items():
        sampler = DomainSampler(sources, mode, 0.5, 16, 42)
        mass = [sum(p for p, s in zip(sampler.probabilities, sources) if s == d) for d in "ABC"]
        np.testing.assert_allclose(mass, expected)
        first = list(sampler)
        assert len(first) == 96 and min(first) >= 0 and max(first) < 100
        assert first == list(sampler)
        if mode == "image":
            assert len(set(first)) == 96
        else:
            generator = torch.Generator().manual_seed(19)
            draws = torch.multinomial(torch.tensor(sampler.probabilities), 50000, True, generator=generator).tolist()
            observed = Counter(sources[index] for index in draws)
            np.testing.assert_allclose([observed[d] / len(draws) for d in "ABC"], expected, atol=0.01)
        sampler.set_epoch(1)
        assert first != list(sampler)
    labels = np.array([[1, 0], [0, 1], [0, 0], [0, 0]], dtype=np.float32)
    uniform = np.full(4, 0.25)
    np.testing.assert_allclose(positive_weights(labels, uniform, "sampling"), [3, 3])
    np.testing.assert_array_equal(positive_weights(labels, uniform, "fixed_image"), [3, 3])
    assert source_from_path("PA100k/example.jpg") == "PA100K"
    print("Sampling probabilities, update budgets, seeds, and loss-weight checks passed")


sampler_self_check()

## 4. Train and record domain-level evidence

The global EVA02-L classifier and optimizer follow the baseline. Training source probabilities are computed from annotations. Each epoch samples exactly `floor(N / BATCH_SIZE) * BATCH_SIZE` images, yielding the same number of optimizer steps across arms. Expected source shares and realized epoch counts are saved. Validation is unresampled and reports each source separately, with equal-source aggregate metrics and flip-logit averaging.

The primary experiment keeps image-distribution positive weights unchanged. This isolates sampling; switching to expected-sampling-distribution weights is explicitly a different experiment. The best mean per-source Challenge_Avg selects the checkpoint. Best validation logits, labels, paths, source names, and attribute order are saved for later comparison; they are excluded from the submission ZIP.

In [ ]:
def build_model(pretrained):
    model = timm.create_model(
        f"{ARCHITECTURE}.{PRETRAINED_TAG}", pretrained=pretrained, num_classes=40,
        img_size=IMAGE_SIZE, drop_path_rate=DROP_PATH,
    )
    nn.init.zeros_(model.head.weight)
    nn.init.zeros_(model.head.bias)
    return model


def parameter_groups(model):
    """AdamW groups with layer-wise learning-rate decay and no decay on norms/biases."""
    depth = len(model.blocks)

    def layer_id(name):
        if name.startswith(("cls_token", "pos_embed", "patch_embed")):
            return 0
        if name.startswith("blocks."):
            return int(name.split(".")[1]) + 1
        return depth + 1  # fc_norm and classifier head.

    groups = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        no_decay = parameter.ndim == 1 or name in ("cls_token", "pos_embed")
        lid = layer_id(name)
        key = (lid, no_decay)
        if key not in groups:
            groups[key] = {
                "params": [],
                "weight_decay": 0.0 if no_decay else WEIGHT_DECAY,
                "lr_scale": LAYER_DECAY ** (depth + 1 - lid),
            }
        groups[key]["params"].append(parameter)
    return list(groups.values())


def challenge_metrics(probabilities, labels, sources):
    """mA, instance F1 and their harmonic mean, averaged over source datasets."""
    eps = 1e-20
    results = []
    per_source = {}
    for source in sorted(set(sources)):
        rows = np.asarray([s == source for s in sources])
        pred = probabilities[rows] > 0.5
        gt = labels[rows] > 0.5
        tp = (pred & gt).sum(0)
        tn = (~pred & ~gt).sum(0)
        positives, negatives = gt.sum(0), (~gt).sum(0)
        label_ma = (tp / (positives + eps) + tn / (negatives + eps)) / 2
        intersection = (pred & gt).sum(1)
        precision = (intersection / (pred.sum(1) + eps)).mean()
        recall = (intersection / (gt.sum(1) + eps)).mean()
        # As in the original evaluation script: F1 of the image-averaged precision and recall.
        ma, f1 = label_ma.mean(), 2 * precision * recall / (precision + recall + eps)
        results.append((ma, f1, 2 * ma * f1 / (ma + f1 + eps)))
        label_f1 = (2 * tp / (pred.sum(0) + positives + eps)).mean()
        instance_acc = (intersection / ((pred | gt).sum(1) + eps)).mean()
        per_source[source] = {
            "mA": float(ma), "Instance_F1": float(f1),
            "Challenge_Avg": float(results[-1][2]), "Label_F1": float(label_f1),
            "Instance_Acc": float(instance_acc), "Instance_Prec": float(precision),
            "Instance_Rec": float(recall),
        }
    ma, f1, avg = np.mean(results, axis=0)
    return {key: float(np.mean([row[key] for row in per_source.values()]))
            for key in next(iter(per_source.values()))}, per_source

In [ ]:
def main():
    if not (DATA_ROOT / "annotations/train.csv").is_file():
        raise FileNotFoundError(
            f"Missing {DATA_ROOT / 'annotations/train.csv'}. Run the download notebook "
            "in this Kaggle session or attach its saved output and set UPAR_DATA_ROOT."
        )
    if not 0 <= WARMUP_EPOCHS < EPOCHS:
        raise ValueError("Require 0 <= WARMUP_EPOCHS < EPOCHS")
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    existing = list(OUTPUT_ROOT.glob(f"{MODEL_TAG}_*"))
    if existing:
        raise FileExistsError(f"Outputs already exist for {MODEL_TAG}; preserve them and change RUN_SUFFIX")
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.backends.cudnn.benchmark = True

    names, train_paths, train_labels = read_annotations(DATA_ROOT / "annotations/train.csv")
    val_names, val_paths, val_labels = read_annotations(DATA_ROOT / "annotations/val.csv")
    if names != val_names:
        raise ValueError("Train and validation attribute orders differ")
    for relative in train_paths[:10] + val_paths[:10]:
        if not (DATA_ROOT / relative).is_file():
            raise FileNotFoundError(f"Missing image {DATA_ROOT / relative}; finish downloading the images")
    val_sources = [source_from_path(path) for path in val_paths]
    train_sources = [source_from_path(path) for path in train_paths]
    if set(train_sources) != {"Market1501", "PA100K", "PETA"}:
        raise ValueError(f"Expected all three training sources, found {set(train_sources)}")
    sampler = DomainSampler(train_sources, SAMPLING_MODE, DOMAIN_MIX, BATCH_SIZE, SEED)
    domain_counts = dict(Counter(train_sources))
    expected_shares = {source: float(sampler.probabilities[np.asarray(train_sources) == source].sum())
                       for source in sorted(domain_counts)}
    print("Training domain counts:", domain_counts)
    print("Expected sampling shares:", expected_shares)
    print(f"Loaded {len(train_paths):,} train and {len(val_paths):,} validation images")

    normalize = transforms.Normalize(MEAN, STD)
    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.Pad(10, padding_mode="edge"),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        normalize,
        transforms.RandomErasing(p=0.25, value="random"),
    ])
    val_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), normalize,
    ])
    train_loader = DataLoader(
        AttributeDataset(DATA_ROOT, train_paths, train_labels, train_transform),
        batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(), drop_last=True,
    )
    val_loader = DataLoader(
        AttributeDataset(DATA_ROOT, val_paths, val_labels, val_transform),
        batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    model = build_model(USE_IMAGENET_WEIGHTS).to(device)
    print(f"{ARCHITECTURE}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
    check_offline_equivalence(model)
    loss_weights = positive_weights(train_labels, sampler.probabilities, LOSS_WEIGHT_MODE)
    pos_weight = torch.tensor(loss_weights, dtype=torch.float32, device=device)
    config = {
        "model_tag": MODEL_TAG, "architecture": ARCHITECTURE, "pretrained_tag": PRETRAINED_TAG,
        "pretrained": USE_IMAGENET_WEIGHTS, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "steps_per_epoch": len(train_loader), "samples_per_epoch": len(sampler),
        "learning_rate": LEARNING_RATE, "layer_decay": LAYER_DECAY, "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": WARMUP_EPOCHS, "drop_path": DROP_PATH, "image_size": IMAGE_SIZE,
        "mean": MEAN, "std": STD, "seed": SEED, "sampling_mode": SAMPLING_MODE,
        "domain_mix": DOMAIN_MIX, "replacement": SAMPLING_MODE != "image",
        "loss_weight_mode": LOSS_WEIGHT_MODE, "positive_weights": loss_weights.tolist(),
        "validation_flip": VALIDATION_FLIP, "domain_counts": domain_counts,
        "expected_domain_shares": expected_shares, "attribute_names": names,
        "versions": {"torch": str(torch.__version__), "timm": timm.__version__, "numpy": np.__version__},
        "annotation_sha256": {split: hashlib.sha256((DATA_ROOT / f"annotations/{split}.csv").read_bytes()).hexdigest()
                              for split in ("train", "val")},
    }
    (OUTPUT_ROOT / f"{MODEL_TAG}_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(parameter_groups(model), lr=LEARNING_RATE)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = WARMUP_EPOCHS * len(train_loader)

    def learning_rate(step):
        if step < warmup_steps:
            return LEARNING_RATE * (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return LEARNING_RATE * 0.5 * (1 + math.cos(math.pi * progress))

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    checkpoint_path = OUTPUT_ROOT / f"{MODEL_TAG}_best.pth"
    best_score = -1.0
    step = 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        sampler.set_epoch(epoch - 1)
        model.train()
        train_total = 0.0
        for images, labels in train_loader:
            for group in optimizer.param_groups:
                group["lr"] = learning_rate(step) * group["lr_scale"]
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(images), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Nonfinite loss at epoch {epoch}, step {step}')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_total += loss.item() * len(images)
            step += 1

        model.eval()
        val_total = 0.0
        val_logits = []
        with torch.inference_mode():
            for images, labels in val_loader:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                    logits = model(images).float()
                    if VALIDATION_FLIP:
                        logits = (logits + model(torch.flip(images, dims=[3])).float()) / 2
                val_total += criterion(logits.float(), labels).item() * len(images)
                val_logits.append(logits.float().cpu().numpy())
        val_logits = np.concatenate(val_logits)
        if not np.isfinite(val_logits).all():
            raise FloatingPointError("Nonfinite validation logits")
        probabilities = torch.from_numpy(val_logits).sigmoid().numpy()
        metrics, per_source = challenge_metrics(probabilities, val_labels, val_sources)
        train_loss = train_total / (len(train_loader) * BATCH_SIZE)
        val_loss = val_total / len(val_paths)
        print(
            f"Epoch {epoch}/{EPOCHS}: train BCE={train_loss:.4f}, val BCE={val_loss:.4f}, "
            f"mA={metrics['mA']:.4f}, Inst_F1={metrics['Instance_F1']:.4f}, "
            f"Challenge_Avg={metrics['Challenge_Avg']:.4f}"
        )
        record = {"epoch": epoch, "train_bce": train_loss, "val_bce": val_loss,
                  "metrics": metrics, "per_source": per_source,
                  "sampled_domain_counts": sampler.last_counts, "optimizer_steps": step}
        history.append(record)
        (OUTPUT_ROOT / f"{MODEL_TAG}_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
        print("Per-source validation:", per_source)
        print("Realized domain draws:", sampler.last_counts)
        if metrics["Challenge_Avg"] > best_score:
            best_score = metrics["Challenge_Avg"]
            torch.save({
                # Stored in float16 to halve the ZIP size; load_state_dict casts back to float32.
                "state_dict": {
                    key: (value.detach().half() if value.is_floating_point() else value.detach()).cpu()
                    for key, value in model.state_dict().items()
                },
                "config": config,
                "architecture": ARCHITECTURE,
                "num_heads": model.blocks[0].attn.num_heads,
                "attribute_names": names,
                "image_size": IMAGE_SIZE,
                "epoch": epoch,
                "val_loss": val_loss,
                "val_metrics": {key: float(value) for key, value in metrics.items()},
            }, checkpoint_path)
            np.savez_compressed(
                OUTPUT_ROOT / f"{MODEL_TAG}_best_val_logits.npz", logits=val_logits,
                labels=val_labels, paths=np.asarray(val_paths), sources=np.asarray(val_sources),
                attribute_names=np.asarray(names), epoch=np.asarray(epoch),
            )
            print(f"Saved {checkpoint_path}")

    # Release training weights and Adam state before loading the packaged models.
    del model, optimizer, scaler, loss, images, labels
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    package_submission(checkpoint_path, [DATA_ROOT / path for path in val_paths[:4]], names)

## 5. Build the inference package

The generated `run.py` exposes the Track 1 API: `load_model()`, `predict_image(sample)` and `predict_batch(samples)`. It defines EVA02 in plain PyTorch (timm is not in the evaluation image), loads the checkpoint, averages the predictions of each image and its horizontal flip, and returns 40 probabilities in the order of `sample["attribute_names"]`. `metadata.yaml` is the unchanged file from `examples/task1/sample_code_submission`. The smoke test also checks that the plain-PyTorch network gives the same logits as the timm model.

In [ ]:
RUN_PY = '''"""UPAR 2027 Track 1 inference entry point (EVA02-L/14).

timm is not installed in the evaluation image, so EVA02 is defined here in plain PyTorch.
Parameter names match timm's Eva, so the fine-tuned state dict loads strictly.
"""
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torchvision import transforms

HERE = Path(__file__).resolve().parent
CHECKPOINT = HERE / "assets/__CHECKPOINT_NAME__"
MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
STD = (0.26862954, 0.26130258, 0.27577711)
BATCH_SIZE = 16
MODEL = None
ATTRIBUTE_NAMES = None
TRANSFORM = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class PatchEmbed(nn.Module):
    def __init__(self, patch_size, dim):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch_size, patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


def rotary_embedding(grid_size, head_dim, ref_grid_size=16, temperature=10000.0):
    """timm RotaryEmbeddingCat (in_pixels=False): sin and cos of 2D patch positions, concatenated."""
    num_bands = head_dim // 4
    bands = 1.0 / temperature ** (torch.arange(num_bands, dtype=torch.float32) / num_bands)
    # EVA02 scales patch coordinates to its 224 / 14 = 16 x 16 pretraining grid.
    t = torch.arange(grid_size, dtype=torch.float32) / grid_size * ref_grid_size
    grid = torch.stack(torch.meshgrid(t, t, indexing="ij"), dim=-1).unsqueeze(-1)
    positions = (grid * bands).reshape(grid_size * grid_size, -1)
    return torch.cat([positions.sin().repeat_interleave(2, -1), positions.cos().repeat_interleave(2, -1)], -1)


def apply_rotary(x, embedding):
    sin, cos = embedding.chunk(2, -1)
    rotated = torch.stack([-x[..., 1::2], x[..., ::2]], -1).reshape(x.shape)  # (x0, x1) -> (-x1, x0)
    return x * cos + rotated * sin


class Attention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, rope):
        b, n, c = x.shape
        q, k, v = (
            projection(x).reshape(b, n, self.num_heads, -1).transpose(1, 2)
            for projection in (self.q_proj, self.k_proj, self.v_proj)
        )
        # Rotary position embeddings rotate the patch tokens only, not the class token.
        q = torch.cat([q[:, :, :1], apply_rotary(q[:, :, 1:], rope)], dim=2).type_as(v)
        k = torch.cat([k[:, :, :1], apply_rotary(k[:, :, 1:], rope)], dim=2).type_as(v)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1_g = nn.Linear(dim, hidden_dim)
        self.fc1_x = nn.Linear(dim, hidden_dim)
        self.act = nn.SiLU()
        self.norm = nn.LayerNorm(hidden_dim, eps=1e-6)
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.norm(self.act(self.fc1_g(x)) * self.fc1_x(x)))


class Block(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = SwiGLU(dim, hidden_dim)

    def forward(self, x, rope):
        x = x + self.attn(self.norm1(x), rope)
        return x + self.mlp(self.norm2(x))


class EVA02(nn.Module):
    """timm Eva for EVA02: RoPE, SwiGLU with sub-LayerNorm, average pooling of patch tokens."""

    def __init__(self, patch_size, num_patches, dim, depth, num_heads, hidden_dim, num_classes):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))
        self.blocks = nn.ModuleList([Block(dim, num_heads, hidden_dim) for _ in range(depth)])
        self.fc_norm = nn.LayerNorm(dim, eps=1e-6)
        self.head = nn.Linear(dim, num_classes)
        self.register_buffer("rope", rotary_embedding(math.isqrt(num_patches), dim // num_heads), persistent=False)

    def forward(self, x):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(len(x), -1, -1), x], dim=1) + self.pos_embed
        for block in self.blocks:
            x = block(x, self.rope)
        return self.head(self.fc_norm(x[:, 1:].mean(dim=1)))


def load_model():
    global MODEL, ATTRIBUTE_NAMES, TRANSFORM
    checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    ATTRIBUTE_NAMES = checkpoint["attribute_names"]
    size = int(checkpoint["image_size"])
    TRANSFORM = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    state = checkpoint["state_dict"]
    model = EVA02(
        patch_size=state["patch_embed.proj.weight"].shape[-1],
        num_patches=state["pos_embed"].shape[1] - 1,  # pos_embed includes the class token.
        dim=state["pos_embed"].shape[2],
        depth=len({key.split(".")[1] for key in state if key.startswith("blocks.")}),
        num_heads=int(checkpoint["num_heads"]),
        hidden_dim=state["blocks.0.mlp.fc1_g.weight"].shape[0],
        num_classes=state["head.weight"].shape[0],
    )
    model.load_state_dict(state)
    MODEL = model.float().to(DEVICE).eval()


def predict_batch(samples):
    if MODEL is None:
        load_model()
    images = []
    for sample in samples:
        with Image.open(sample["image_path"]) as image:
            images.append(TRANSFORM(image.convert("RGB")))
    batch = torch.stack(images).to(DEVICE)
    with torch.inference_mode(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        logits = MODEL(batch).float() + MODEL(torch.flip(batch, dims=[3])).float()
    probabilities = (logits / 2).sigmoid().clamp(0.0, 1.0).cpu().numpy()
    if not np.isfinite(probabilities).all():
        raise FloatingPointError('Nonfinite inference probabilities')
    rows = []
    for sample, row in zip(samples, probabilities):
        order = [ATTRIBUTE_NAMES.index(name) for name in sample["attribute_names"]]
        rows.append(row[order].astype(float).tolist())
    return rows


def predict_image(sample):
    return predict_batch([sample])[0]
'''

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls predict_image() / predict_batch(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(checkpoint_path, sample_images, names):
    folder = OUTPUT_ROOT / f"{MODEL_TAG}_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    (folder / "run.py").write_text(RUN_PY.replace("__CHECKPOINT_NAME__", checkpoint_path.name).replace("BATCH_SIZE = 16", f"BATCH_SIZE = {INFERENCE_BATCH_SIZE}"), encoding="utf-8")
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    shutil.copy2(checkpoint_path, assets / checkpoint_path.name)
    archive_path = OUTPUT_ROOT / f"{MODEL_TAG}_submission.zip"
    expected = {"run.py", "metadata.yaml", f"assets/{checkpoint_path.name}"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            spec = importlib.util.spec_from_file_location("upar_submission_check", Path(temp_dir) / "run.py")
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            module.load_model()
            samples = [
                {"index": i, "image": str(path), "image_path": str(path), "attribute_names": names}
                for i, path in enumerate(sample_images)
            ]
            batch = np.asarray(module.predict_batch(samples))
            single = np.asarray(module.predict_image(samples[0]))
            assert batch.shape == (len(samples), 40) and single.shape == (40,)
            assert np.isfinite(batch).all() and ((batch >= 0) & (batch <= 1)).all()
            np.testing.assert_allclose(single, batch[0], atol=1e-3, rtol=1e-3)
            reordered = dict(samples[0], attribute_names=list(reversed(names)))
            np.testing.assert_allclose(module.predict_image(reordered), single[::-1], atol=1e-3, rtol=1e-3)
            print("Packaged inference shape, range, single/batch and attribute-order checks passed")
            # run.py re-implements EVA02; it must match the timm model exactly.
            reference = build_model(False).to(module.DEVICE).eval()
            reference.load_state_dict(module.MODEL.state_dict())
            inputs = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=module.DEVICE)
            with torch.inference_mode():
                difference = (reference(inputs) - module.MODEL(inputs)).abs().max().item()
            assert difference < 1e-3, f"run.py and timm outputs differ by {difference}"
            print(f"run.py matches timm (max logit difference {difference:.1e})")
    print("Checkpoint:", checkpoint_path)
    print(f"ZIP size: {archive_path.stat().st_size / 1e6:.0f} MB")
    print("Upload this ZIP to Codabench:", archive_path)

def check_offline_equivalence(model):
    """Fail before expensive training if pinned timm and offline inference differ."""
    namespace = {"__file__": str(OUTPUT_ROOT / "preflight_run.py")}
    exec(compile(RUN_PY, "preflight_run.py", "exec"), namespace)
    state = model.state_dict()
    device = next(model.parameters()).device
    # Restore RNG and zero classifiers after a meaningful nonzero-head check.
    devices = [device.index or 0] if device.type == "cuda" else []
    training = model.training
    saved_weight = model.head.weight.detach().clone()
    saved_bias = model.head.bias.detach().clone()
    offline = None
    try:
        with torch.random.fork_rng(devices=devices), torch.no_grad():
            torch.manual_seed(123)
            nn.init.normal_(model.head.weight, std=0.01)
            nn.init.normal_(model.head.bias, std=0.01)
            offline = namespace["EVA02"](
                patch_size=14, num_patches=(IMAGE_SIZE // 14) ** 2,
                dim=state["pos_embed"].shape[2], depth=len(model.blocks),
                num_heads=model.blocks[0].attn.num_heads,
                hidden_dim=state["blocks.0.mlp.fc1_g.weight"].shape[0], num_classes=40,
            ).to(device).eval()
            offline.load_state_dict(model.state_dict(), strict=True)
            model.eval()
            images = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
            reference, actual = model(images).float(), offline(images).float()
            if not torch.isfinite(actual).all():
                raise FloatingPointError("Offline preflight returned nonfinite logits")
            torch.testing.assert_close(actual, reference, atol=1e-3, rtol=1e-3)
            print("Pre-training offline/timm check passed; max difference:", (actual - reference).abs().max().item())
    finally:
        with torch.no_grad():
            model.head.weight.copy_(saved_weight)
            model.head.bias.copy_(saved_bias)
        model.train(training)
        del offline
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

## 6. Run training and create the ZIP

Run the final cell to train one arm. Download all mode-prefixed outputs before changing settings. The saved configuration records exact domain counts, expected sampling probabilities by domain, loss weights, versions, annotation hashes, and the update budget. History records realized counts and source validation metrics each epoch. Best logits and checkpoint share the selected epoch.

For a comparison, run `image` and `mixture` with otherwise identical settings in separate fresh kernels. Use `image_replacement` if you need to separate replacement from domain reweighting. Do not use the jointly observed hidden-test scores to optimize mixture weights. Confirm promising changes with another seed and source-held-out evaluation before attributing transfer gains to domain balancing.

Only the offline inference entry point, unchanged metadata, and selected checkpoint enter the submission ZIP. Full experiment records and validation labels remain outside it. The package uses the same global EVA02-L architecture, normalization, and flip-logit averaging as the baseline; it does not need source identities at inference. The final cell also compares the packaged plain-PyTorch model with timm.

In [ ]:
main()